Assignment 3
Compare BERT to interpretable classifiers for text classification

1. Download the ReviewBase dataset
Split dotat on space character
The data set also contains a class labael associated with each text

In [1]:
file_training = open('ReviewBaseTraining.txt', 'r', encoding = 'utf-8')
training_data_string = file_training.read()
file_training.close()

file_validation = open('ReviewBaseValidation.txt', 'r', encoding = 'utf-8')
validation_data_string = file_validation.read()
file_validation.close()

file_test = open('ReviewBaseTest.txt', 'r', encoding = 'utf-8')
test_data_string = file_test.read()
file_test.close()


In [2]:
training_data = training_data_string.splitlines()
validation_data = validation_data_string.splitlines()
test_data = test_data_string.splitlines()




def split_labels(data):
    current_label = None
    x = []
    y = []
    for word in data:
        label, text = word.split("\t")
        x.append(text)
        y.append(label)
    return x, y

x_train, y_train = split_labels(training_data)
x_val, y_val = split_labels(validation_data)
x_test, y_test = split_labels(test_data)

y_train = [int(label) for label in y_train]
y_val = [int(label) for label in y_val]
y_test = [int(label) for label in y_test]

print(x_train[:2])
print(y_train[:2])



["Story of a man who has unnatural feelings for a pig . Starts out with a opening scene that is a terrific example of absurd comedy . A formal orchestra audience is turned into an insane , violent mob by the crazy chantings of it ' s singers . Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting . Even those from the era should be turned off . The cryptic dialogue would make Shakespeare seem easy to a third grader . On a technical level it ' s better than you might think with some good cinematography by future great Vilmos Zsigmond . Future stars Sally Kirkland and Frederic Forrest can be seen briefly .", 'Bromwell High is a cartoon comedy . It ran at the same time as some other programs about school life , such as " Teachers " . My 35 years in the teaching profession lead me to believe that Bromwell High \' s satire is much closer to reality than is " Teachers " . The scramble to survive financially , the insightful students 

Set up and fine-tune the DeBERTa-v3 model using low-rank adaptiation(LoRA) 
Use a single fully connected layer (after transformer bloacks)


In [3]:
!pip install -q transformers[sentencepiece] accelerate

!pip install peft

In [4]:
print(type(x_train))
print(type(x_train[0]))
print("Example text:", x_train[0][:120])

print(type(y_train))
print("Example label:", y_train[0])

<class 'list'>
<class 'str'>
Example text: Story of a man who has unnatural feelings for a pig . Starts out with a opening scene that is a terrific example of absu
<class 'list'>
Example label: 0


In [5]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from peft import LoraConfig, get_peft_model

from torch.utils.data import Dataset, DataLoader

c:\Users\Min Dator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#Fine tuning involves setting suitable values for the weights int he feedforward layer
#Slightly modifying weights in BERT itself
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

train_data = [(x, y) for x, y in zip(x_train, y_train)]
test_data = [(x, y) for x, y in zip(x_test, y_test)]
validation_data = [(x, y) for x, y in zip(x_val, y_val)]

dataloaded_train = DataLoader(train_data, batch_size = 4, shuffle = True)


data_loader_test = DataLoader(test_data, batch_size = 4, shuffle = False)
data_loader_val = DataLoader(validation_data, batch_size = 4, shuffle = False)




lora_config = LoraConfig(r = 8, target_modules = ["query_proj", "key_proj", "value_proj"])

model = get_peft_model(model, lora_config)

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear) and ("attn" in name.lower() or "attention" in name.lower()):
        print(name)




Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1285.74it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.

base_model.model.deberta.encoder.layer.0.attention.self.query_proj.base_layer
base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_A.default
base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_B.default
base_model.model.deberta.encoder.layer.0.attention.self.key_proj.base_layer
base_model.model.deberta.encoder.layer.0.attention.self.key_proj.lora_A.default
base_model.model.deberta.encoder.layer.0.attention.self.key_proj.lora_B.default
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.base_layer
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_A.default
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_B.default
base_model.model.deberta.encoder.layer.0.attention.output.dense
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.base_layer
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_A.default
base_model.model.deberta.encoder.layer.1.attention.self.

In [23]:
batch = next(iter(dataloaded_train))
print("type(batch):", type(batch))
print("len(batch):", len(batch))

texts, labels = batch
print("type(texts):", type(texts), "len:", len(texts))
print("type(labels):", type(labels), "len:", len(labels))

print("first text:", texts[0][:80])
print("first label:", labels[0])

type(batch): <class 'list'>
len(batch): 2
type(texts): <class 'tuple'> len: 32
type(labels): <class 'torch.Tensor'> len: 32
first text: I didn ' t think it could be done , but something has come along and replaced Op
first label: tensor(0)


cuda available: False
device: cpu
model device: cpu


In [9]:

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

epochs = 2

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    steps = 0
    for batch in dataloaded_train:
        texts, labels = batch     

        train_x_token = tokenizer(list(texts), return_tensors="pt", truncation = True, padding = True, max_length = 128).to(device)

        labels_t = torch.tensor(labels, dtype=torch.long).to(device)
        #optimizer = "hddell"
        #Actual training
        out = model(input_ids = train_x_token["input_ids"], labels = labels_t, attention_mask = train_x_token["attention_mask"])
        print("shlw")
        loss = out.loss
    

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        steps += 1



print(f"epoch {epoch+1}, avg_loss = {running_loss/steps}")






C:\Users\Min Dator\AppData\Local\Temp\ipykernel_12948\1393704500.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_t = torch.tensor(labels, dtype=torch.long).to(device)


shlw


KeyboardInterrupt: 

In [ ]:
for param in bert.parameters():
    param.requires_grad = False  #Freezes BERT

class BERT_architecture(nn.Module):

In [ ]:
model = make_pipeline(
    CountVectorizer(ngram_range=(1,2)),
    MultinomialNB()
)

vectorizer = CountVectorizer(ngram_range=(1,2))
x_train_vectorized = vectorizer.fit_transform(x_train)
model.fit(x_train, y_train)

#Good idea to implement the perceptron rule myself

